In [1]:
import os
import cv2

In [2]:
import openslide

In [3]:
from lxml import etree

In [4]:
import numpy as np
import matplotlib.pyplot as plt

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, auc

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [8]:
def xml_to_tumor_mask(xml_path, slide, level = 5):

    dims = slide.level_dimensions[level]
    downsample = slide.level_downsamples[level]

    mask = np.zeros([dims[1], dims[0]], dtype = np.uint8)
    tree = etree.parse(xml_path)
    root = tree.getroot()
    annotations = root.findall(".//Annotation")

    for annotation in annotations:
        coords = []
        for c in annotation.findall(".//Coordinate"):
            x = float(c.get("X")) / downsample
            y = float(c.get("Y")) / downsample
            coords.append([int(x), int(y)])

        if len(coords) > 2:
            pts = np.array(coords, dtype = np.int32)
            cv2.fillPoly(mask, [pts], 1)
    return mask      

In [9]:
def extract_tumor_patches_from_slide(slide_path, xml_path, 
                               patch_size = 256, stride = 256,
                                     mask_level = 5):

    slide = openslide.OpenSlide(slide_path)

    mask = xml_to_tumor_mask(xml_path, slide, level = mask_level)
    downsample = slide.level_downsamples[mask_level]

    width, height = slide.dimensions
    patches = []

    for y in range(0, height - patch_size, stride):
        for x in range(0, width - patch_size, stride):

            patch = slide.read_region((x,y), 0, (patch_size, patch_size))
            patch = np.array(patch)[:, :, :3]

            if patch.mean() > 200 or patch.mean() < 30:
                continue

            mask_x = int(x / downsample)
            mask_y = int(y / downsample)

            if mask_x >= mask.shape[1] or mask_y >= mask.shape[0]:

                label = 0

            else:

                label = int(mask[mask_y, mask_x])

            if label == 1:
                
                patches.append(patch)
                          
    slide.close()
    total_found = len(patches)
    print(f"{os.path.basename(slide_path)}: found {total_found} tumor patches")
        
    labels = np.ones(len(patches), dtype = np.float32)

    return patches, labels

In [10]:
def extract_normal_patches_from_slide(slide_path, 
                               patch_size = 256, stride = 512,
                                     max_normal_patches = 5000):
    

    slide = openslide.OpenSlide(slide_path)
    
    width, height = slide.dimensions
    patches = []

    for y in range(0, height - patch_size, stride):
        for x in range(0, width - patch_size, stride):

            patch = slide.read_region((x,y), 0, (patch_size, patch_size))
            patch = np.array(patch)[:, :, :3]

            if patch.mean() > 200 or patch.mean() < 30:
                continue

            patches.append(patch)
    
    slide.close()
    total_found = len(patches)

    if total_found > max_normal_patches:
        indices = np.random.choice(total_found, max_normal_patches, replace = False)
        patches = np.array([patches[i] for i in indices])
        print(f"{os.path.basename(slide_path)}: Found {total_found} normal patches, kept{max_normal_patches}")

    else:

        print(f"{os.path.basename(slide_path)}: Found {total_found} normal patches, kept{total_found}")
        patches = np.array(patches)
        
    labels = np.zeros(len(patches), dtype = np.float32)
    
    return patches, labels

In [11]:
DATA_DIR = r"C:\Users\sasan\OneDrive\Desktop\Pathology_Images"

In [12]:
tumor_slides = [
    os.path.join(DATA_DIR, "tumor_016.tif"),
    os.path.join(DATA_DIR, "tumor_017.tif"),
    os.path.join(DATA_DIR, "tumor_018.tif"),
    os.path.join(DATA_DIR, "tumor_022.tif"),
    os.path.join(DATA_DIR, "tumor_023.tif"),
    os.path.join(DATA_DIR, "tumor_024.tif"),
    os.path.join(DATA_DIR, "tumor_025.tif"),
    os.path.join(DATA_DIR, "tumor_026.tif"),
    os.path.join(DATA_DIR, "tumor_027.tif"),
    os.path.join(DATA_DIR, "tumor_030.tif"),
    os.path.join(DATA_DIR, "tumor_031.tif"),
    os.path.join(DATA_DIR, "tumor_032.tif"),
    os.path.join(DATA_DIR, "tumor_033.tif"),
    os.path.join(DATA_DIR, "tumor_034.tif"),
    os.path.join(DATA_DIR, "tumor_035.tif"),
    os.path.join(DATA_DIR, "tumor_036.tif"),
    os.path.join(DATA_DIR, "tumor_037.tif"),
    os.path.join(DATA_DIR, "tumor_038.tif"),
    os.path.join(DATA_DIR, "tumor_039.tif"),
    os.path.join(DATA_DIR, "tumor_040.tif"),
    os.path.join(DATA_DIR, "tumor_041.tif"),
    os.path.join(DATA_DIR, "tumor_042.tif"),
    os.path.join(DATA_DIR, "tumor_045.tif"),
    os.path.join(DATA_DIR, "tumor_057.tif"),
    os.path.join(DATA_DIR, "tumor_058.tif"),
    os.path.join(DATA_DIR, "tumor_059.tif"),
    os.path.join(DATA_DIR, "tumor_060.tif"),
    os.path.join(DATA_DIR, "tumor_064.tif"),
    os.path.join(DATA_DIR, "tumor_065.tif"),
    os.path.join(DATA_DIR, "tumor_070.tif"),
    os.path.join(DATA_DIR, "tumor_071.tif"),
    os.path.join(DATA_DIR, "tumor_091.tif"),
]

In [13]:
tumor_xmls = [
    os.path.join(DATA_DIR, "tumor_016.xml"),
    os.path.join(DATA_DIR, "tumor_017.xml"),
    os.path.join(DATA_DIR, "tumor_018.xml"),
    os.path.join(DATA_DIR, "tumor_022.xml"),
    os.path.join(DATA_DIR, "tumor_023.xml"),
    os.path.join(DATA_DIR, "tumor_024.xml"),
    os.path.join(DATA_DIR, "tumor_025.xml"),
    os.path.join(DATA_DIR, "tumor_026.xml"),
    os.path.join(DATA_DIR, "tumor_027.xml"),
    os.path.join(DATA_DIR, "tumor_030.xml"),
    os.path.join(DATA_DIR, "tumor_031.xml"),
    os.path.join(DATA_DIR, "tumor_032.xml"),
    os.path.join(DATA_DIR, "tumor_033.xml"),
    os.path.join(DATA_DIR, "tumor_034.xml"),
    os.path.join(DATA_DIR, "tumor_035.xml"),
    os.path.join(DATA_DIR, "tumor_036.xml"),
    os.path.join(DATA_DIR, "tumor_037.xml"),
    os.path.join(DATA_DIR, "tumor_038.xml"),
    os.path.join(DATA_DIR, "tumor_039.xml"),
    os.path.join(DATA_DIR, "tumor_040.xml"),
    os.path.join(DATA_DIR, "tumor_041.xml"),
    os.path.join(DATA_DIR, "tumor_042.xml"),
    os.path.join(DATA_DIR, "tumor_045.xml"),
    os.path.join(DATA_DIR, "tumor_057.xml"),
    os.path.join(DATA_DIR, "tumor_058.xml"),
    os.path.join(DATA_DIR, "tumor_059.xml"),
    os.path.join(DATA_DIR, "tumor_060.xml"),
    os.path.join(DATA_DIR, "tumor_064.xml"),
    os.path.join(DATA_DIR, "tumor_065.xml"),
    os.path.join(DATA_DIR, "tumor_070.xml"),
    os.path.join(DATA_DIR, "tumor_071.xml"),
    os.path.join(DATA_DIR, "tumor_091.xml"),
]

In [14]:
normal_slides = [
    os.path.join(DATA_DIR, "normal_094.tif"),
    os.path.join(DATA_DIR, "normal_095.tif"),
    os.path.join(DATA_DIR, "normal_096.tif"),
    os.path.join(DATA_DIR, "normal_097.tif"),
    os.path.join(DATA_DIR, "normal_098.tif"),
    os.path.join(DATA_DIR, "normal_099.tif"),
    os.path.join(DATA_DIR, "normal_108.tif"),
    os.path.join(DATA_DIR, "normal_132.tif"),
    os.path.join(DATA_DIR, "normal_133.tif"),
    os.path.join(DATA_DIR, "normal_134.tif"),
    os.path.join(DATA_DIR, "normal_135.tif"),
    os.path.join(DATA_DIR, "normal_136.tif"),
    os.path.join(DATA_DIR, "normal_143.tif"),
]

In [15]:
print(f"Tumor slides: {len(tumor_slides)}")
print(f"Normal slides: {len(normal_slides)}")

Tumor slides: 32
Normal slides: 13


In [16]:
all_tumor_patches = []
all_tumor_labels = []

for slide, xml in zip(tumor_slides, tumor_xmls):

    patches, labels = extract_tumor_patches_from_slide(slide, xml,
                               patch_size = 256, stride = 256, 
                                                       mask_level = 5
                                                      )

    all_tumor_patches.append(patches)
    all_tumor_labels.append(labels)

tumor_016.tif: found 2506 tumor patches
tumor_017.tif: found 2 tumor patches
tumor_018.tif: found 768 tumor patches
tumor_022.tif: found 76 tumor patches
tumor_023.tif: found 38 tumor patches
tumor_024.tif: found 17 tumor patches
tumor_025.tif: found 1291 tumor patches
tumor_026.tif: found 10695 tumor patches
tumor_027.tif: found 40 tumor patches
tumor_030.tif: found 5 tumor patches
tumor_031.tif: found 2047 tumor patches
tumor_032.tif: found 49 tumor patches
tumor_033.tif: found 1726 tumor patches
tumor_034.tif: found 960 tumor patches
tumor_035.tif: found 3 tumor patches
tumor_036.tif: found 1225 tumor patches
tumor_037.tif: found 537 tumor patches
tumor_038.tif: found 553 tumor patches
tumor_039.tif: found 3075 tumor patches
tumor_040.tif: found 9 tumor patches
tumor_041.tif: found 44 tumor patches
tumor_042.tif: found 418 tumor patches
tumor_045.tif: found 188 tumor patches
tumor_057.tif: found 26 tumor patches
tumor_058.tif: found 5351 tumor patches
tumor_059.tif: found 4 tumor pa

In [17]:
X_train_tumor = np.concatenate(all_tumor_patches)
y_train_tumor = np.concatenate(all_tumor_labels)

In [18]:
np.save("X_train_tumor.npy", X_train_tumor)
np.save("y_train_tumor.npy", y_train_tumor)

In [19]:
print(len(X_train_tumor))
print(len(y_train_tumor))

34374
34374


In [20]:
all_normal_patches = []
all_normal_labels = []

for slide in normal_slides:

    patches, labels = extract_normal_patches_from_slide(slide,
                                                 patch_size = 256, stride = 512, 
                                                       max_normal_patches = 2700
                                                       )

    all_normal_patches.append(patches)
    all_normal_labels.append(labels)

normal_094.tif: Found 3341 normal patches, kept2700
normal_095.tif: Found 9871 normal patches, kept2700
normal_096.tif: Found 5190 normal patches, kept2700
normal_097.tif: Found 10735 normal patches, kept2700
normal_098.tif: Found 3103 normal patches, kept2700
normal_099.tif: Found 12444 normal patches, kept2700
normal_108.tif: Found 1274 normal patches, kept1274
normal_132.tif: Found 7333 normal patches, kept2700
normal_133.tif: Found 5770 normal patches, kept2700
normal_134.tif: Found 2613 normal patches, kept2613
normal_135.tif: Found 9121 normal patches, kept2700
normal_136.tif: Found 9070 normal patches, kept2700
normal_143.tif: Found 8616 normal patches, kept2700


In [21]:
X_train_normal = np.concatenate(all_normal_patches)
y_train_normal = np.concatenate(all_normal_labels)

In [22]:
np.save("X_train_normal.npy", X_train_normal)
np.save("y_train_normal.npy", y_train_normal)

In [23]:
print(len(X_train_normal))
print(len(y_train_normal))

33587
33587


In [24]:
X_train = np.concatenate([X_train_tumor, X_train_normal])
y_train = np.concatenate([y_train_tumor, y_train_normal])

In [25]:
print(f' Total patches: {len(X_train)}, Total tumor patches: {y_train.sum()}, Total normal patches: {(y_train == 0).sum()}')

 Total patches: 67961, Total tumor patches: 34374.0, Total normal patches: 33587


In [26]:
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
print("Saved!")

Saved!


In [ ]:
# load saved train patches
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
print(f"Loaded train patches: {len(X_train)}")

In [28]:
print(f"\n── Train Patch Summary ──")
print(f"  Total train patches: {len(X_train)}")
print(f"  Patch shape:         {X_train[0].shape}")
print(f"  Pixel range:         {X_train.min():.0f} to {X_train.max():.0f}")
print(f"  Memory size:         {X_train.nbytes / 1e9:.2f} GB")


── Train Patch Summary ──
  Total train patches: 67961
  Patch shape:         (256, 256, 3)
  Pixel range:         0 to 255
  Memory size:         13.36 GB


In [29]:
ANNOTATIONS_DIR = r"C:\Users\sasan\OneDrive\Desktop\Pathology_Images\lesion_annotations"

annotated = os.listdir(ANNOTATIONS_DIR)
patients = ["patient_015", "patient_017", "patient_020", "patient_024", "patient_046", "patient_086"]

for patient in patients:
    print(f"\n{patient}:")
    for f in sorted(annotated):
        if patient.split("_")[1] in f:
            print(f"  {f}")


patient_015:
  patient_015_node_1.xml
  patient_015_node_2.xml

patient_017:
  patient_017_node_1.xml
  patient_017_node_2.xml
  patient_017_node_4.xml

patient_020:
  patient_020_node_2.xml
  patient_020_node_4.xml

patient_024:
  patient_024_node_1.xml
  patient_024_node_2.xml

patient_046:
  patient_046_node_3.xml
  patient_046_node_4.xml

patient_086:
  patient_086_node_0.xml
  patient_086_node_4.xml


In [30]:
CAMELYON17_DIR   = r"C:\Users\sasan\OneDrive\Desktop\Pathology_Images"
ANNOTATIONS_DIR  = os.path.join(CAMELYON17_DIR, "lesion_annotations")

In [31]:
# Tumor slides with XML annotations (macro/micro only)
val_tumor_slides = [
    os.path.join(CAMELYON17_DIR, "patient_015", "patient_015_node_1.tif"),
    os.path.join(CAMELYON17_DIR, "patient_015", "patient_015_node_2.tif"),
    os.path.join(CAMELYON17_DIR, "patient_017", "patient_017_node_2.tif"),
    os.path.join(CAMELYON17_DIR, "patient_017", "patient_017_node_4.tif"),
    os.path.join(CAMELYON17_DIR, "patient_020", "patient_020_node_2.tif"),
    os.path.join(CAMELYON17_DIR, "patient_020", "patient_020_node_4.tif"),
    os.path.join(CAMELYON17_DIR, "patient_046", "patient_046_node_3.tif"),
    os.path.join(CAMELYON17_DIR, "patient_046", "patient_046_node_4.tif"),
]

In [32]:
val_tumor_xmls = [
    os.path.join(ANNOTATIONS_DIR, "patient_015_node_1.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_015_node_2.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_017_node_2.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_017_node_4.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_020_node_2.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_020_node_4.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_046_node_3.xml"),
    os.path.join(ANNOTATIONS_DIR, "patient_046_node_4.xml"),
]

In [33]:
# Normal slides (negative nodes only)
val_normal_slides = [
    os.path.join(CAMELYON17_DIR, "patient_015", "patient_015_node_0.tif"),
    os.path.join(CAMELYON17_DIR, "patient_015", "patient_015_node_3.tif"),
    os.path.join(CAMELYON17_DIR, "patient_017", "patient_017_node_0.tif"),
    os.path.join(CAMELYON17_DIR, "patient_020", "patient_020_node_0.tif"),
    os.path.join(CAMELYON17_DIR, "patient_020", "patient_020_node_3.tif"),
    os.path.join(CAMELYON17_DIR, "patient_024", "patient_024_node_0.tif"),
    os.path.join(CAMELYON17_DIR, "patient_024", "patient_024_node_3.tif"),
    os.path.join(CAMELYON17_DIR, "patient_046", "patient_046_node_0.tif"),
    os.path.join(CAMELYON17_DIR, "patient_046", "patient_046_node_1.tif"),
    os.path.join(CAMELYON17_DIR, "patient_046", "patient_046_node_2.tif"),
    os.path.join(CAMELYON17_DIR, "patient_086", "patient_086_node_1.tif"),
    os.path.join(CAMELYON17_DIR, "patient_086", "patient_086_node_2.tif"),
    os.path.join(CAMELYON17_DIR, "patient_086", "patient_086_node_3.tif"),
]

In [34]:
print("Checking tumor slides and xmls:")
for s, x in zip(val_tumor_slides, val_tumor_xmls):
    print(f"  {os.path.exists(s)} {os.path.exists(x)} — {os.path.basename(s)} — {os.path.basename(x)}")

print("\nChecking normal slides:")
for s in val_normal_slides:
    print(f"  {os.path.exists(s)} — {os.path.basename(s)}")

Checking tumor slides and xmls:
  True True — patient_015_node_1.tif — patient_015_node_1.xml
  True True — patient_015_node_2.tif — patient_015_node_2.xml
  True True — patient_017_node_2.tif — patient_017_node_2.xml
  True True — patient_017_node_4.tif — patient_017_node_4.xml
  True True — patient_020_node_2.tif — patient_020_node_2.xml
  True True — patient_020_node_4.tif — patient_020_node_4.xml
  True True — patient_046_node_3.tif — patient_046_node_3.xml
  True True — patient_046_node_4.tif — patient_046_node_4.xml

Checking normal slides:
  True — patient_015_node_0.tif
  True — patient_015_node_3.tif
  True — patient_017_node_0.tif
  True — patient_020_node_0.tif
  True — patient_020_node_3.tif
  True — patient_024_node_0.tif
  True — patient_024_node_3.tif
  True — patient_046_node_0.tif
  True — patient_046_node_1.tif
  True — patient_046_node_2.tif
  True — patient_086_node_1.tif
  True — patient_086_node_2.tif
  True — patient_086_node_3.tif


In [35]:
# ── Val tumor extraction ──
val_tumor_patches = []
val_tumor_labels  = []

for slide, xml in zip(val_tumor_slides, val_tumor_xmls):
    patches, labels = extract_tumor_patches_from_slide(slide, xml,
                                                       patch_size=256, stride=256,
                                                       mask_level=5
                                                      )
    val_tumor_patches.append(patches)
    val_tumor_labels.append(labels)

patient_015_node_1.tif: found 1166 tumor patches
patient_015_node_2.tif: found 260 tumor patches
patient_017_node_2.tif: found 2019 tumor patches
patient_017_node_4.tif: found 141 tumor patches
patient_020_node_2.tif: found 35 tumor patches
patient_020_node_4.tif: found 729 tumor patches
patient_046_node_3.tif: found 234 tumor patches
patient_046_node_4.tif: found 48 tumor patches


In [36]:
X_val_tumor = np.concatenate(val_tumor_patches)
y_val_tumor = np.concatenate(val_tumor_labels)

In [37]:
np.save("X_val_tumor.npy", X_val_tumor)
np.save("y_val_tumor.npy", y_val_tumor)
print(f"Val tumor saved: {len(X_val_tumor)} patches")

Val tumor saved: 4632 patches


In [ ]:
X_val_tumor = np.load("X_val_tumor.npy")
y_val_tumor = np.load("y_val_tumor.npy")
print(f"Val tumor loaded: {len(X_val_tumor)} patches")

In [38]:
# ── Val normal extraction ──
val_normal_patches = []
val_normal_labels  = []

for slide in val_normal_slides:
    patches, labels = extract_normal_patches_from_slide(slide,
                                                        patch_size=256, stride=512, 
                                                        max_normal_patches = 370
                                                       )
    val_normal_patches.append(patches)
    val_normal_labels.append(labels)

patient_015_node_0.tif: Found 1971 normal patches, kept370
patient_015_node_3.tif: Found 3606 normal patches, kept370
patient_017_node_0.tif: Found 961 normal patches, kept370
patient_020_node_0.tif: Found 5076 normal patches, kept370
patient_020_node_3.tif: Found 4638 normal patches, kept370
patient_024_node_0.tif: Found 14050 normal patches, kept370
patient_024_node_3.tif: Found 5752 normal patches, kept370
patient_046_node_0.tif: Found 9574 normal patches, kept370
patient_046_node_1.tif: Found 4033 normal patches, kept370
patient_046_node_2.tif: Found 4699 normal patches, kept370
patient_086_node_1.tif: Found 5149 normal patches, kept370
patient_086_node_2.tif: Found 4335 normal patches, kept370
patient_086_node_3.tif: Found 1440 normal patches, kept370


In [39]:
X_val_normal = np.concatenate(val_normal_patches)
y_val_normal = np.concatenate(val_normal_labels)

In [40]:
np.save("X_val_normal.npy", X_val_normal)
np.save("y_val_normal.npy", y_val_normal)
print(f"Val normal saved: {len(X_val_normal)} patches")

Val normal saved: 4810 patches


In [ ]:
X_val_normal = np.load("X_val_normal.npy")
y_val_normal = np.load("y_val_normal.npy")
print(f"Val normal loaded: {len(X_val_normal)} patches")

In [41]:
# ── Combine ──
X_val = np.concatenate([X_val_tumor, X_val_normal])
y_val = np.concatenate([y_val_tumor, y_val_normal])

In [42]:
np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)

In [43]:
print(f"\nVal Total: {len(X_val)}, Tumor: {int(y_val.sum())}, Normal: {int((y_val==0).sum())}")


Val Total: 9442, Tumor: 4632, Normal: 4810


In [ ]:
X_val   = np.load("X_val.npy")
y_val   = np.load("y_val.npy")
print(f"Loaded val patches: {len(X_val)} patches")

In [44]:
print(f"\n── Train Patch Summary ──")
print(f"  Total val patches: {len(X_val)}")
print(f"  Patch shape:         {X_val[0].shape}")
print(f"  Pixel range:         {X_val.min():.0f} to {X_val.max():.0f}")
print(f"  Memory size:         {X_val.nbytes / 1e9:.2f} GB")


── Train Patch Summary ──
  Total val patches: 9442
  Patch shape:         (256, 256, 3)
  Pixel range:         0 to 255
  Memory size:         1.86 GB
